In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import MultipleLocator
from math import sqrt
from numpy import concatenate
from pandas import read_csv, DataFrame, concat
from datetime import datetime
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import MinMaxScaler, StandardScaler

import keras.backend as K
from keras.layers import Multiply
from keras.models import Sequential ,Model
from keras.layers import Dense , Input , Reshape , Flatten ,Permute , Lambda , RepeatVector ,Conv1D , MaxPooling1D , Dropout, Bidirectional, Activation
from keras.layers import GRU, LSTM
from keras.utils.vis_utils import plot_model
from keras.optimizers import SGD,Adam
from keras.utils import np_utils   #np_utils
from keras.callbacks import TensorBoard  #TensorBoard可视化

Using TensorFlow backend.


In [2]:
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,mean_absolute_percentage_error
def RMSE(valid_y,pre_Y):
    XP1 = valid_y.copy()
    XA1 = pre_Y.copy()
    print('RMSE(sklearn):',np.sqrt(mean_squared_error(XP1, XA1)))

In [3]:
# 验证
# data1 = pd.read_csv('surgeN_自回归步长1.csv')
# surge_yuan = np.array(data1)[:, 0:1]
# surge1_1 = np.array(data1)[:, 1:2]
# evaluate(surge_yuan,surge1_1)

In [4]:
# T:0
# H:1
# Surge:2
# Heave:4
# Force1:8
data = pd.read_csv('t_2_11.2_50.csv')
data.head()

,T,H,Surge,Sway,Heave,Roll,Pitch,Yaw,Force1,Force2
0,0.1,-422.0,-0.0175,0.000289,-72.6,-1.160000e-09,-8.240000e-08,-1.350000e-09,4910.0,4910.0
1,0.2,-402.0,-0.2550,0.000348,-277.0,-1.430000e-07,2.830000e-03,-1.320000e-07,4690.0,4650.0
2,0.3,-380.0,1.2500,-0.035900,-413.0,-2.880000e-06,1.940000e-02,2.190000e-07,4650.0,4380.0
3,0.4,-356.0,4.1700,-0.070800,-339.0,-1.250000e-06,2.620000e-02,1.160000e-06,4790.0,4420.0
4,0.5,-330.0,3.0200,-0.097500,-242.0,4.580000e-07,2.310000e-02,2.060000e-06,4880.0,4550.0


In [5]:
data.describe()

,T,H,Surge,Sway,Heave,Roll,Pitch,Yaw,Force1,Force2
count,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.00000,40000.000000
mean,2000.050030,0.163048,5.607178,0.042822,-24.376800,-0.000003,-0.000050,0.000002,4994.70075,4957.142000
std,1154.714989,401.008617,93.005630,2.851035,379.528224,0.000376,0.018356,0.000222,753.57593,277.992156
min,0.100000,-1510.000000,-373.000000,-11.100000,-1390.000000,-0.001810,-0.067500,-0.001290,2290.00000,4120.000000
25%,1000.075015,-278.000000,-56.400000,-1.830000,-285.000000,-0.000251,-0.012900,-0.000136,4480.00000,4750.000000
50%,2000.050030,-5.955000,4.280000,0.030350,-31.600000,-0.000003,0.000474,-0.000001,4980.00000,4950.000000
75%,3000.025045,277.000000,64.700000,1.900000,236.000000,0.000246,0.012600,0.000135,5500.00000,5170.000000
max,4000.000060,1340.000000,452.000000,11.600000,1190.000000,0.001830,0.068400,0.001310,8110.00000,5700.000000


In [6]:
data_distance = np.hstack((np.array(data)[:, 1:8], np.array(data)[:, 8:10]))
print(data_distance)
print(data_distance.shape)

[[-4.22e+02 -1.75e-02  2.89e-04 ... -1.35e-09  4.91e+03  4.91e+03]
 [-4.02e+02 -2.55e-01  3.48e-04 ... -1.32e-07  4.69e+03  4.65e+03]
 [-3.80e+02  1.25e+00 -3.59e-02 ...  2.19e-07  4.65e+03  4.38e+03]
 ...
 [-2.13e+02 -1.20e+02 -5.24e-01 ...  2.37e-04  4.16e+03  5.16e+03]
 [-2.31e+02 -1.22e+02 -2.43e-01 ...  2.07e-04  4.14e+03  5.17e+03]
 [-2.49e+02 -1.22e+02 -6.28e-01 ...  8.97e-05  4.12e+03  5.17e+03]]
(40000, 9)


In [7]:
# H_scaler = MinMaxScaler(feature_range=(-1, 1))
# H = H_scaler.fit_transform(data_distance[:,0:1])
# Surge_scaler = MinMaxScaler(feature_range=(-1, 1))
# Surge = Surge_scaler.fit_transform(data_distance[:,1:2])
# # Sway_scaler = MinMaxScaler(feature_range=(-1, 1))
# # Sway = Sway_scaler.fit_transform(data_distance[:,2:3])
# Heave_scaler = MinMaxScaler(feature_range=(-1, 1))
# Heave = Heave_scaler.fit_transform(data_distance[:,3:4])
# # Roll_scaler = MinMaxScaler(feature_range=(-1, 1))
# # Roll = Roll_scaler.fit_transform(data_distance[:,4:5]*1e6)
# Pitch_scaler = MinMaxScaler(feature_range=(-1, 1))
# Pitch = Pitch_scaler.fit_transform(data_distance[:,5:6])
# Yaw_scaler = MinMaxScaler(feature_range=(-1, 1))
# Yaw = Yaw_scaler.fit_transform(data_distance[:,6:7]*1e6)
Force1_scaler = MinMaxScaler(feature_range=(-1, 1))
Force1 = Force1_scaler.fit_transform(data_distance[:,7:8])
Force2_scaler = MinMaxScaler(feature_range=(-1, 1))
Force2 = Force2_scaler.fit_transform(data_distance[:,8:9])
# zong_data = np.hstack(())

In [8]:
data_force11 = pd.read_csv('单_force1_步长1.csv')
data_force12 = pd.read_csv('多_force1_步长1.csv')


data_force21 = pd.read_csv('单_force2_步长1.csv')
data_force22 = pd.read_csv('多_force2_步长1.csv')

In [9]:
# 反归一化a=-1,b=1
# b-a=2；Hmax-Hmin=np.max(data_distance[:,0:1])-np.min(data_distance[:,0:1])
# guiyihua(输入步长)_变量(输出步长)

# guiyihua_H=2*(data_distance[:,0:1]-np.min(data_distance[:,0:1]))/(np.max(data_distance[:,0:1])-np.min(data_distance[:,0:1]))-1
# print(guiyihua_H)

# force1  data_distance[:,7:8]
yuan_force1 = 2*(np.array(data_force11)[:, 0:1]-np.min(data_distance[:,7:8]))/(np.max(data_distance[:,7:8])-np.min(data_distance[:,7:8]))-1
guiyihua1_force1 = 2*(np.array(data_force11)[:, 1:2]-np.min(data_distance[:,7:8]))/(np.max(data_distance[:,7:8])-np.min(data_distance[:,7:8]))-1
guiyihua2_force1 = 2*(np.array(data_force12)[:, 1:2]-np.min(data_distance[:,7:8]))/(np.max(data_distance[:,7:8])-np.min(data_distance[:,7:8]))-1


# force2  data_distance[:,8:9]
yuan_force2 = 2*(np.array(data_force21)[:, 0:1]-np.min(data_distance[:,8:9]))/(np.max(data_distance[:,8:9])-np.min(data_distance[:,8:9]))-1
guiyihua1_force2 = 2*(np.array(data_force21)[:, 1:2]-np.min(data_distance[:,8:9]))/(np.max(data_distance[:,8:9])-np.min(data_distance[:,8:9]))-1
guiyihua2_force2 = 2*(np.array(data_force22)[:, 1:2]-np.min(data_distance[:,8:9]))/(np.max(data_distance[:,8:9])-np.min(data_distance[:,8:9]))-1

In [10]:
RMSE(yuan_force1,guiyihua1_force1)
RMSE(yuan_force1,guiyihua2_force1)
print('-------------')
RMSE(yuan_force2,guiyihua1_force2)
RMSE(yuan_force2,guiyihua2_force2)

RMSE(sklearn): 0.03350005996145681
RMSE(sklearn): 0.03348456902890447
-------------
RMSE(sklearn): 0.06281261814070528
RMSE(sklearn): 0.06355724870913938


In [11]:
def R(valid_y,pre_Y):
    XP1 = valid_y.copy()
    XA1 = pre_Y.copy()
    real1 = np.trapz(abs((XP1 - (XP1.sum()/(XP1.shape[0])))).reshape(XP1.shape[0],), dx=0.1)
    pre1 = np.trapz(abs((XA1 - (XP1.sum()/(XP1.shape[0])))).reshape(XA1.shape[0],), dx=0.1)
    Acc1 = 1 - abs(1 - (pre1/real1))
    print('Acc:', Acc1)

In [12]:
R(yuan_force1,guiyihua1_force1)
R(yuan_force1,guiyihua2_force1)
print('-------------')
R(yuan_force2,guiyihua1_force2)
R(yuan_force2,guiyihua2_force2)

Acc: 0.9748046738003849
Acc: 0.9861356180371837
-------------
Acc: 0.9832809677541121
Acc: 0.9936947212190335
